<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ crime buffer mapping ★ </h3>
  <p> This notebook maps crime data points to edges using various buffer types.</p>
</div>

In [100]:
import osmnx as ox
import os
import networkx as nx
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely import wkt
import math
import warnings
warnings.filterwarnings("ignore")
import plotly.express as px
from shapely.geometry import MultiLineString, LineString
import branca

In [101]:
def cb_mapping(
    source,
    mode=3,
    buf=25,
    keep_buffers=False,
    id_col='edge_id'
):
    """
    Maps crime points to street edges using one of three strategies.

    Parameters
    ----------
    source : pandas.DataFrame or geopandas.GeoDataFrame
        Edge table with a 'geometry' column (WKT or shapely) and an ID column.
    mode : int
        1 = direct nearest (snap points to nearest edge)
        2 = buffer edges, intersect points
        3 = buffer points, intersect edges  (allows multi-edge mapping)
    buf : float
        Buffer size in projected CRS units (meters when using UTM).
    keep_buffers : bool
        If False in mode 3, output geometry is the (original) crime point geometry,
        and temporary buffer geometry is dropped. If True, edge geometry remains active.
    id_col : str
        Edge ID column used to identify successful joins and for output.

    Returns
    -------
    geopandas.GeoDataFrame
        Rows representing mapped crimes (possibly multiple edges per crime in mode 3).
        Contains a boolean 'is_day' and retains edge attributes. Geometry depends on mode/keep_buffers.
    """

    import os
    import numpy as np
    import pandas as pd
    import geopandas as gpd
    from shapely import wkt
    from shapely.geometry import Point, LineString, Polygon

    # ---------- Helpers ----------
    def _safe_unique_ids(df: pd.DataFrame, col: str) -> np.ndarray:
        """Return unique, non-null IDs from column `col`, or empty array if missing."""
        if col in df.columns:
            vals = df[col].dropna()
            try:
                return vals.astype(int).unique()
            except Exception:
                return vals.unique()
        return np.array([], dtype=int)

    # ---------- Read/normalize edges ----------
    if isinstance(source, gpd.GeoDataFrame):
        edges_gdf = source.copy()
        if edges_gdf.geometry.name != "geometry":
            edges_gdf = edges_gdf.set_geometry(edges_gdf.geometry.name)
        if edges_gdf.crs is None:
            edges_gdf.set_crs("EPSG:4326", inplace=True)
    else:
        edges = source.copy()
        if id_col not in edges.columns:
            edges = edges.reset_index().rename(columns={'index': id_col})

        # Detect whether geometry column already contains shapely objects vs WKT
        first_geom_val = edges["geometry"].dropna().iloc[0] if not edges["geometry"].dropna().empty else None
        if isinstance(first_geom_val, (Point, LineString, Polygon)):
            edges_gdf = gpd.GeoDataFrame(edges, geometry=edges["geometry"], crs="EPSG:4326")
        else:
            edges_gdf = gpd.GeoDataFrame(edges, geometry=gpd.GeoSeries.from_wkt(edges["geometry"]), crs="EPSG:4326")

    # ---------- Read crimes (robust to cwd) ----------
    base = os.getcwd()
    day_path = os.path.join(base, "..", "crime", "data", "crime_points", "crime_daytime.csv")
    night_path = os.path.join(base, "..", "crime", "data", "crime_points", "crime_nighttime.csv")
    crime_day = pd.read_csv(day_path)
    crime_night = pd.read_csv(night_path)

    crime_day_gdf = gpd.GeoDataFrame(
        crime_day, geometry=gpd.points_from_xy(crime_day.Longitude, crime_day.Latitude), crs='EPSG:4326'
    )
    crime_night_gdf = gpd.GeoDataFrame(
        crime_night, geometry=gpd.points_from_xy(crime_night.Longitude, crime_night.Latitude), crs='EPSG:4326'
    )

    # ---------- Project to UTM for accurate buffering ----------
    utm_crs = edges_gdf.estimate_utm_crs()
    edges_proj = edges_gdf.to_crs(utm_crs)
    crime_day_proj = crime_day_gdf.to_crs(utm_crs)
    crime_night_proj = crime_night_gdf.to_crs(utm_crs)

    # ---------- Mode-specific joins ----------
    if mode == 1:
        # Direct nearest snapping of all crimes (1-to-1)
        join_day = gpd.sjoin_nearest(
            crime_day_proj, edges_proj, how='left',
            max_distance=100, distance_col="dist_to_edge"
        )
        join_night = gpd.sjoin_nearest(
            crime_night_proj, edges_proj, how='left',
            max_distance=100, distance_col="dist_to_edge"
        )
        join_day["is_day"] = True
        join_night["is_day"] = False
        join_all = pd.concat([join_day, join_night], ignore_index=True)
        print(f"Finished mapping (mode=1, buf={buf}): {join_all.shape[0]} mappings")
        return join_all

    elif mode == 2:
        # Buffer edges, intersect crimes (multi crimes per edge; 1 edge per intersect)
        edges_buf = edges_proj.copy()
        edges_buf = edges_buf.set_geometry(edges_buf.geometry.buffer(buf))
        join_day = gpd.sjoin(crime_day_proj, edges_buf, how='left', predicate='intersects')
        join_night = gpd.sjoin(crime_night_proj, edges_buf, how='left', predicate='intersects')

    elif mode == 3:
        # Buffer crimes, intersect edges (multi edges per crime)
        day_pts = crime_day_proj.copy()
        night_pts = crime_night_proj.copy()

        day_buf = day_pts.set_geometry(day_pts.geometry.buffer(buf))
        night_buf = night_pts.set_geometry(night_pts.geometry.buffer(buf))

        # Left join edges with crime buffers -> index_right are original crime indices
        join_day = gpd.sjoin(edges_proj, day_buf, how='left', predicate='intersects')
        join_night = gpd.sjoin(edges_proj, night_buf, how='left', predicate='intersects')

        print("rows in day: ", join_day.shape[0], join_day.columns)
        print("rows in night: ", join_night.shape[0], join_night.columns)

    else:
        raise ValueError("mode must be 1, 2, or 3.")

    # ---------- Ensure ID columns exist (idempotent) ----------
    for df in (join_day, join_night):
        if 'Original_Crime_ID_Day' not in df.columns:
            df['Original_Crime_ID_Day'] = np.nan
        if 'Original_Crime_ID_Night' not in df.columns:
            df['Original_Crime_ID_Night'] = np.nan

    # ---------- Mode-specific population of Original_Crime_ID_* BEFORE dropping helpers ----------
    if mode == 2:
        # join is crimes LEFT JOIN edges_buf, so the row index is the original crime index
        valid_day_rows = join_day[id_col].notna() if id_col in join_day.columns else pd.Series(False, index=join_day.index)
        join_day.loc[valid_day_rows, 'Original_Crime_ID_Day'] = join_day.loc[valid_day_rows].index.values

        valid_night_rows = join_night[id_col].notna() if id_col in join_night.columns else pd.Series(False, index=join_night.index)
        join_night.loc[valid_night_rows, 'Original_Crime_ID_Night'] = join_night.loc[valid_night_rows].index.values

    elif mode == 3:
        # join is edges LEFT JOIN crime buffers -> index_right are original crime indices
        if 'index_right' in join_day.columns:
            valid = join_day['index_right'].notna()
            join_day.loc[valid, 'Original_Crime_ID_Day'] = join_day.loc[valid, 'index_right'].values
        if 'index_right' in join_night.columns:
            valid = join_night['index_right'].notna()
            join_night.loc[valid, 'Original_Crime_ID_Night'] = join_night.loc[valid, 'index_right'].values

    # ---------- Geometry handling for mode 3 ----------
    if mode == 3:
        if not keep_buffers:
            # Build geometry_point from original crime points using the preserved index_right
            # Only construct if we still have index_right
            original_point_geometry_day = pd.Series(index=join_day.index, dtype='object')
            original_point_geometry_night = pd.Series(index=join_night.index, dtype='object')

            if 'index_right' in join_day.columns:
                valid_idx_day = join_day.loc[join_day['index_right'].notna(), 'index_right'].astype(crime_day_proj.index.dtype)
                original_point_geometry_day.loc[join_day['index_right'].notna()] = crime_day_proj.loc[valid_idx_day, crime_day_proj.geometry.name].values

            if 'index_right' in join_night.columns:
                valid_idx_night = join_night.loc[join_night['index_right'].notna(), 'index_right'].astype(crime_night_proj.index.dtype)
                original_point_geometry_night.loc[join_night['index_right'].notna()] = crime_night_proj.loc[valid_idx_night, crime_night_proj.geometry.name].values

            join_day['geometry_point'] = original_point_geometry_day
            join_night['geometry_point'] = original_point_geometry_night

            # Switch active geometry to the original crime point; drop edge geom and helper col
            join_day = join_day.set_geometry('geometry_point')
            join_night = join_night.set_geometry('geometry_point')
            join_day = join_day.drop(columns=['geometry', 'index_right'], errors='ignore')
            join_night = join_night.drop(columns=['geometry', 'index_right'], errors='ignore')
        else:
            # keep edge geometry as active; just drop helper if present
            join_day = join_day.drop(columns=['index_right'], errors='ignore')
            join_night = join_night.drop(columns=['index_right'], errors='ignore')

    # ---------- Label time of day ----------
    join_day["is_day"] = True
    join_night["is_day"] = False

    # ---------- Compute mapped IDs safely (pre-leftovers) ----------
    mapped_crime_ids_day = _safe_unique_ids(join_day, 'Original_Crime_ID_Day')
    mapped_crime_ids_night = _safe_unique_ids(join_night, 'Original_Crime_ID_Night')

    # ---------- Leftover (unmapped) crimes -> snap nearest within buf ----------
    unmapped_day_crime_gdf = crime_day_proj[~crime_day_proj.index.isin(mapped_crime_ids_day)]
    unmapped_night_crime_gdf = crime_night_proj[~crime_night_proj.index.isin(mapped_crime_ids_night)]

    final_join_day = join_day.copy()
    final_join_night = join_night.copy()

    # Unmapped DAY
    if not unmapped_day_crime_gdf.empty:
        nd_day = gpd.sjoin_nearest(
            unmapped_day_crime_gdf, edges_proj, how="left",
            max_distance=buf, distance_col="dist_to_edge"
        )
        nd_day_mapped = nd_day.dropna(subset=[id_col]) if id_col in nd_day.columns else nd_day.iloc[0:0]
        if not nd_day_mapped.empty:
            nd_day_mapped['Original_Crime_ID_Day'] = nd_day_mapped.index
            # ensure all columns exist so we can align/concat
            for col in final_join_day.columns:
                if col not in nd_day_mapped.columns:
                    nd_day_mapped[col] = np.nan
            nd_day_mapped = nd_day_mapped[final_join_day.columns]
            final_join_day = pd.concat([final_join_day, nd_day_mapped], ignore_index=True)

    # Unmapped NIGHT
    if not unmapped_night_crime_gdf.empty:
        nd_night = gpd.sjoin_nearest(
            unmapped_night_crime_gdf, edges_proj, how="left",
            max_distance=buf, distance_col="dist_to_edge"
        )
        nd_night_mapped = nd_night.dropna(subset=[id_col]) if id_col in nd_night.columns else nd_night.iloc[0:0]
        if not nd_night_mapped.empty:
            nd_night_mapped['Original_Crime_ID_Night'] = nd_night_mapped.index
            for col in final_join_night.columns:
                if col not in nd_night_mapped.columns:
                    nd_night_mapped[col] = np.nan
            nd_night_mapped = nd_night_mapped[final_join_night.columns]
            final_join_night = pd.concat([final_join_night, nd_night_mapped], ignore_index=True)

    # ---------- Combine day + night ----------
    join_all = pd.concat([final_join_day, final_join_night], ignore_index=True)

    # ---------- Cleanup ----------
    drop_cols = [
        'Offense ID', 'Latitude', 'Longitude', 'Offense Date', 'Hour', 'pt_geom',
        'Original_Crime_ID_Day', 'Original_Crime_ID_Night'  # temporary IDs
    ]
    join_all.drop(columns=[c for c in drop_cols if c in join_all.columns], inplace=True, errors='ignore')

    # Final geometry choice
    if mode == 3 and not keep_buffers:
        if 'geometry_point' in join_all.columns:
            join_all = join_all.set_geometry('geometry_point')
            join_all.drop(columns=['pt_geom'], errors='ignore', inplace=True)
    elif 'geometry' in join_all.columns:
        join_all = join_all.set_geometry('geometry')

    print(f"Finished mapping (mode={mode}, buf={buf}): {join_all.shape[0]} mappings")
    return join_all


<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ multiple mapping ★ </h3>
  <p> use the code below to map for multiple buffer types at once. buffer sizes can be modified. </p>
</div>

In [102]:
def cb_mapping_multiple():

    # read edges in from csv
    edges = pd.read_csv('data/edges/edges_seattle_filtered.csv')
    edges['geometry'] = edges['geometry'].apply(wkt.loads)

    # direct "mapping" = nearest (mode 1)
    result = cb_mapping(edges, 1, 0)
    result.to_csv("data/mappings/edges_with_safety_score_direct_mapping.csv")

    # # street buffers
    for d in (10, 15, 20, 25):
        result = cb_mapping(edges, 2, d)
        result.to_csv(f"data/mappings/edges_with_safety_score_street_buffer_{d}m.csv")

    # crime point buffers
    for d in (10, 15, 20, 25, 40, 50):
        result = cb_mapping(edges, 3, d)
        result.to_csv(f"data/mappings/edges_with_safety_score_point_buffer_{d}m.csv")
    # d = 10
    # result = cb_mapping(edges, 3, d)
    # result.to_csv(f"data/mappings/edges_with_safety_score_point_buffer_{d}m.csv")

In [103]:
cb_mapping_multiple()

Finished mapping (mode=1, buf=0): 23549 mappings
Finished mapping (mode=2, buf=10): 59531 mappings
Finished mapping (mode=2, buf=15): 128693 mappings
Finished mapping (mode=2, buf=20): 154704 mappings
Finished mapping (mode=2, buf=25): 169188 mappings
rows in day:  40682 Index(['edge_id', 'u', 'v', 'key', 'osmid', 'highway', 'oneway', 'reversed',
       'length', 'lanes', 'maxspeed', 'name', 'access', 'service', 'geometry',
       'bridge', 'tunnel', 'ref', 'index_right', 'Offense ID', 'Offense Date',
       'Latitude', 'Longitude', 'Offense DateTime', 'Offense Type',
       'Crime Score', 'Hour'],
      dtype='object')
rows in night:  42828 Index(['edge_id', 'u', 'v', 'key', 'osmid', 'highway', 'oneway', 'reversed',
       'length', 'lanes', 'maxspeed', 'name', 'access', 'service', 'geometry',
       'bridge', 'tunnel', 'ref', 'index_right', 'Offense ID', 'Offense Date',
       'Latitude', 'Longitude', 'Offense DateTime', 'Offense Type',
       'Crime Score', 'Hour'],
      dtype='obj